In [8]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [9]:
daily_series = pd.read_csv("C:/Users/ellis/OneDrive - Newcastle University/SESDP/Final Rainfall Datasets/WETTERtimeseries.csv", index_col="Date")
daily_series

,Amount
Date,
01/01/2025,0.066781
02/01/2025,2.609644
03/01/2025,0.378507
04/01/2025,23.842416
05/01/2025,2.364699
...,...
27/12/2099,0.901811
28/12/2099,0.225275
29/12/2099,0.000000


In [10]:
test = pd.read_parquet("C:/Users/ellis/OneDrive - Newcastle University/SESDP/Pond Simulation/ParquetFilesAEP/31-01-2025.parquet")

In [11]:
def plot_flow_range(df: pd.DataFrame, start_date: str, end_date: str, title: str, volume_col: str = 'Pond Volume'):
    """
    Plot Pond Volume (secondary y-axis, fixed 0–127 m³) alongside Inflow and Outflow (primary y-axis)
    for a range of days from start_date to end_date inclusive, using the Plotly 'presentation' template.

    Parameters
    ----------
    df : pd.DataFrame
        Time-series DataFrame with a DateTimeIndex and columns 'Inflow', 'Outflow', 'Pond Volume'.
    start_date : str
        The start date in 'YYYY-MM-DD' format.
    end_date : str
        The end date in 'YYYY-MM-DD' format.
    volume_col : str, default 'Pond Volume'
        Column name for pond volume.
    """
    # Ensure datetime index
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.copy()
        df.index = pd.to_datetime(df.index)

    # Parse and normalize bounds
    start_dt = pd.to_datetime(start_date).normalize()
    end_dt = pd.to_datetime(end_date).normalize()

    # Build mask: include all times within the date range
    mask = (df.index.normalize() >= start_dt) & (df.index.normalize() <= end_dt)
    if not mask.any():
        raise ValueError(f"No data between {start_dt.date()} and {end_dt.date()}.")
    df_sel = df.loc[mask]

    # Check needed columns
    for col in ['Inflow', 'Outflow', volume_col]:
        if col not in df_sel.columns:
            raise KeyError(f"Column not found: {col}")

    # Build figure with two y-axes
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Primary axis traces: Inflow & Outflow
    fig.add_trace(
        go.Scatter(x=df_sel.index, y=df_sel['Inflow'],  name='Inflow',  mode='lines'),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=df_sel.index, y=df_sel['Outflow'], name='Outflow', mode='lines'),
        secondary_y=False
    )

    # Secondary axis trace: Pond Volume
    fig.add_trace(
        go.Scatter(x=df_sel.index, y=df_sel[volume_col], name=volume_col, mode='lines'),
        secondary_y=True
    )

    # Layout adjustments
    fig.update_layout(
        template='presentation',
        title=title,
        xaxis_title="Time",
        legend_title="Series",
        width=1200,
        height=800
    )
    fig.update_yaxes(title_text="Flow (m³/s)", secondary_y=False)
    fig.update_yaxes(title_text="Volume (m³)", secondary_y=True, range=[0, 127])
    return fig

fig0 = plot_flow_range(test, "2025-02-01", "2025-02-01", title= "Retention Pond Perfromance for 100-year Design Storm")
fig0.write_image("C:/Users/ellis/OneDrive - Newcastle University/SESDP/Pond Simulation/Figures/100year.png")

In [12]:
def plot_flow_range(
    df: pd.DataFrame,
    start_date: str,
    end_date: str,
    title: str,
    volume_col: str = 'Pond Volume'
) -> go.Figure:
    """
    Plot Pond Volume (secondary y-axis, fixed 0–127 m³) alongside Inflow and Outflow (primary y-axis)
    for a range of days from start_date to end_date inclusive, using the Plotly 'presentation' template.

    Parameters
    ----------
    df : pd.DataFrame
        Time-series DataFrame with a DateTimeIndex and columns 'Inflow', 'Outflow', 'Pond Volume'.
    start_date : str
        The start date in 'YYYY-MM-DD' format.
    end_date : str
        The end date in 'YYYY-MM-DD' format.
    title : str
        The title of the plot.
    volume_col : str, default 'Pond Volume'
        Column name for pond volume.
    
    Returns
    -------
    fig : plotly.graph_objects.Figure
        The Plotly Figure object.
    """
    # Ensure datetime index
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.copy()
        df.index = pd.to_datetime(df.index)

    # Parse and normalize bounds
    start_dt = pd.to_datetime(start_date).normalize()
    end_dt   = pd.to_datetime(end_date).normalize()

    # Build mask: include all times within the date range
    mask = (df.index.normalize() >= start_dt) & (df.index.normalize() <= end_dt)
    if not mask.any():
        raise ValueError(f"No data between {start_dt.date()} and {end_dt.date()}.")
    df_sel = df.loc[mask]

    # Check required columns
    for col in ['Inflow', 'Outflow', volume_col]:
        if col not in df_sel.columns:
            raise KeyError(f"Column not found: {col}")

    # Build figure with two y-axes
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Primary axis traces: Inflow & Outflow
    fig.add_trace(
        go.Scatter(x=df_sel.index, y=df_sel['Inflow'],  name='Inflow',  mode='lines'),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=df_sel.index, y=df_sel['Outflow'], name='Outflow', mode='lines'),
        secondary_y=False
    )

    # Secondary axis trace: Pond Volume
    fig.add_trace(
        go.Scatter(x=df_sel.index, y=df_sel[volume_col], name=volume_col, mode='lines'),
        secondary_y=True
    )

    # Layout adjustments
    fig.update_layout(
        template='presentation',
        title=title,
        xaxis_title="Time",
        legend_title="Series",
        width=2400,
        height=800
    )
    fig.update_yaxes(title_text="Flow (m³/s)",    secondary_y=False)
    fig.update_yaxes(title_text="Volume (m³)",    secondary_y=True, range=[0, 127])

    return fig


fig0 = plot_flow_range(
    test,
    start_date="2025-02-01",
    end_date="2025-02-01",
    title="Retention Pond Performance for 100-year Design Storm"
)

# Save as PNG
fig0.write_image(
    r"C:\Users\ellis\OneDrive - Newcastle University\SESDP\Pond Simulation\Figures\100year.png"
)


In [13]:
def plot_pres_flow_range(
    df: pd.DataFrame,
    start_date: str,
    end_date: str,
    title: str,
    volume_col: str = 'Pond Volume'
) -> go.Figure:
    """
    Plot pond volume on the primary y-axis alongside inflow, outflow,
    and (if present) overflow on the secondary y-axis, over a specified
    date range. Time is shown in hours from the start of the event,
    with the original styling and figure dimensions preserved.

    Parameters
    ----------
    df : pd.DataFrame
        Time-series DataFrame with a DateTimeIndex and at least
        'Inflow', 'Outflow', and the specified volume_col.
    start_date : str
        Start date in 'YYYY-MM-DD' format.
    end_date : str
        End date in 'YYYY-MM-DD' format.
    title : str
        Plot title (include any storm details here).
    volume_col : str, default 'Pond Volume'
        Column name for pond volume.
    
    Returns
    -------
    fig : plotly.graph_objects.Figure
    """
    # -- Prepare data subset --
    df_proc = df.copy()
    # Drop rainfall column if present
    df_proc = df_proc.drop(columns=['Rainfall Amount'], errors='ignore')

    # Ensure datetime index
    if not pd.api.types.is_datetime64_any_dtype(df_proc.index):
        df_proc.index = pd.to_datetime(df_proc.index)

    # Parse and normalize date bounds
    start_dt = pd.to_datetime(start_date).normalize()
    end_dt   = pd.to_datetime(end_date).normalize()
    mask = (df_proc.index.normalize() >= start_dt) & (df_proc.index.normalize() <= end_dt)
    if not mask.any():
        raise ValueError(f"No data between {start_dt.date()} and {end_dt.date()}.")
    df_sel = df_proc.loc[mask]

    # -- Determine last nonzero volume time --
    nz = df_sel[df_sel[volume_col] > 0].index
    last_time = nz[-1] if len(nz) > 0 else df_sel.index[0]

    # Trim to end at last nonzero volume
    df_trim = df_sel[df_sel.index <= last_time].copy()
    start_time = df_trim.index[0].normalize()
    df_trim['Time'] = (df_trim.index - start_time).total_seconds() / 3600  # hours

    # -- Build figure --
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Primary axis: Pond Volume
    fig.add_trace(
        go.Scatter(
            x=df_trim['Time'],
            y=df_trim[volume_col],
            name=volume_col,
            mode='lines'
        ),
        secondary_y=False
    )

    # Secondary axis: Inflow & Outflow
    fig.add_trace(
        go.Scatter(
            x=df_trim['Time'],
            y=df_trim['Inflow'],
            name='Inflow',
            mode='lines'
        ),
        secondary_y=True
    )
    fig.add_trace(
        go.Scatter(
            x=df_trim['Time'],
            y=df_trim['Outflow'],
            name='Outflow',
            mode='lines'
        ),
        secondary_y=True
    )

    # Optional overflow trace
    if 'Overflow' in df_trim.columns:
        fig.add_trace(
            go.Scatter(
                x=df_trim['Time'],
                y=df_trim['Overflow'],
                name='Overflow',
                mode='lines'
            ),
            secondary_y=True
        )

    last_hours = (last_time - start_time).total_seconds() / 3600

    # -- Apply original styling & layout --
    fig.update_layout(
        title=title,
        xaxis_title="Time (hours from start of event)",
        plot_bgcolor="#FFFFFF",
        paper_bgcolor="#FFE7E1",
        font=dict(color="#3E675D", size=16),
        title_font=dict(color="#3E675D", size=20),
        showlegend=True,
        width=1200,
        height=800
    )

    fig.update_yaxes(
        title_text=f"{volume_col} (m³)",
        range=[0, df_trim[volume_col].max()],
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey',
        tickfont=dict(color="#3E675D", size=14),
        secondary_y=False
    )

    fig.update_yaxes(
        title_text="Flow Rate (m³/s)",
        range=[0, 0.12],
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey',
        tickfont=dict(color="#3E675D", size=14),
        secondary_y=True
    )

    fig.update_xaxes(
        range=[0, 6],
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey',
        tickfont=dict(color="#3E675D", size=18)
    )

    return fig

fig1 = plot_pres_flow_range(
    test,
    start_date="2025-02-01",
    end_date="2025-02-01",
    title="Retention Pond Performance for 100-year Design Storm"
)

In [14]:
def plot_report_flow_range(
    df: pd.DataFrame,
    start_date: str,
    end_date: str,
    title: str,
    volume_col: str = 'Pond Volume'
) -> go.Figure:
    """
    Plot pond volume on the primary y-axis alongside inflow, outflow,
    and (if present) overflow on the secondary y-axis, over a specified
    date range—using white backgrounds and black text throughout.

    Parameters
    ----------
    df : pd.DataFrame
        Time-series DataFrame with a DateTimeIndex and at least
        'Inflow', 'Outflow', and the specified volume_col.
    start_date : str
        Start date in 'YYYY-MM-DD' format.
    end_date : str
        End date in 'YYYY-MM-DD' format.
    title : str
        Plot title.
    volume_col : str, default 'Pond Volume'
        Column name for pond volume.
    
    Returns
    -------
    fig : plotly.graph_objects.Figure
    """
    # Copy & drop rainfall column if present
    df_proc = df.drop(columns=['Rainfall Amount'], errors='ignore').copy()

    # Ensure datetime index
    if not pd.api.types.is_datetime64_any_dtype(df_proc.index):
        df_proc.index = pd.to_datetime(df_proc.index)

    # Normalize bounds and filter
    start_dt = pd.to_datetime(start_date).normalize()
    end_dt   = pd.to_datetime(end_date).normalize()
    mask = (df_proc.index.normalize() >= start_dt) & (df_proc.index.normalize() <= end_dt)
    if not mask.any():
        raise ValueError(f"No data between {start_dt.date()} and {end_dt.date()}.")
    df_sel = df_proc.loc[mask]

    # Find last nonzero-volume point
    nz = df_sel[df_sel[volume_col] > 0].index
    last_time = nz[-1] if len(nz) > 0 else df_sel.index[0]

    # Trim beyond last nonzero, compute hours
    df_trim = df_sel[df_sel.index <= last_time].copy()
    start_time = df_trim.index[0].normalize()
    df_trim['Time'] = (df_trim.index - start_time).total_seconds() / 3600

    # Create subplot with secondary y
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Pond Volume on primary y-axis
    fig.add_trace(
        go.Scatter(
            x=df_trim['Time'],
            y=df_trim[volume_col],
            name=volume_col,
            mode='lines'
        ),
        secondary_y=False
    )

    # Inflow & Outflow on secondary y-axis
    fig.add_trace(
        go.Scatter(x=df_trim['Time'], y=df_trim['Inflow'],  name='Inflow',  mode='lines'),
        secondary_y=True
    )
    fig.add_trace(
        go.Scatter(x=df_trim['Time'], y=df_trim['Outflow'], name='Outflow', mode='lines'),
        secondary_y=True
    )

    # Optional Overflow
    if 'Overflow' in df_trim.columns:
        fig.add_trace(
            go.Scatter(x=df_trim['Time'], y=df_trim['Overflow'], name='Overflow', mode='lines'),
            secondary_y=True
        )

    last_hours = (last_time - start_time).total_seconds() / 3600

    # Layout with white backgrounds and black text
    fig.update_layout(
        title=title,
        xaxis_title="Time (hours from start of event)",
        plot_bgcolor="white",
        paper_bgcolor="white",
        font=dict(color="black", size=16),
        title_font=dict(color="black", size=20),
        showlegend=True,
        width=1200,
        height=400
    )

    # Primary y-axis style
    fig.update_yaxes(
        title_text=f"{volume_col} (m³)",
        range=[0, df_trim[volume_col].max()],
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey',
        tickfont=dict(color="black", size=14),
        secondary_y=False
    )

    # Secondary y-axis style
    fig.update_yaxes(
        title_text="Flow Rate (m³/s)",
        range=[0, 0.12],
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey',
        tickfont=dict(color="black", size=14),
        secondary_y=True
    )

    # X-axis style
    fig.update_xaxes(
        range=[0, 6],
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey',
        tickfont=dict(color="black", size=18)
    )

    return fig

fig2 = plot_report_flow_range(
    test,
    start_date="2025-02-01",
    end_date="2025-02-01",
    title="Retention Pond Performance for 100-year Design Storm"
)